# Acervo que Fala — Notebook 04 (v6): o sistema redesenhado

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

Este notebook testa uma hipótese levantada pelo autor antes do julgamento do bake-off: **parte dos erros não vinha do modelo, e sim de instruções mal desenhadas**. A análise confirmou três mecanismos (documentados em `avaliacao/analise_prompt_rubrica.md` no repositório): frases de exemplo do prompt copiadas literalmente para a saída ("sobre a argila bege" apareceu numa bolsa de fio de tucum), perguntas que induzem resposta na observação ("incluindo bordas, faixas e acabamentos"), e palavras-gatilho lidas sem a negação ("não há close-up" → alt marcado como detalhe).

**O que a v6 muda — o sistema, não o modelo (mesmo Qwen3-VL-8B):**

- **Observação v3 em seções nomeadas** (OBJETO, MATERIAIS E CORES, PADRÕES…, ENQUADRAMENTO, ARTEFATOS), com contexto do acervo e uma guarda explícita: reconhecer materiais, nunca adivinhar significado;
- **Redação v9 com Contrato de Fontes**: sem fonte → não escreve (omitir é sempre permitido); incerteza herda-se da seção LEGIBILIDADE; divergência vira flag, nunca harmonização; exemplos só com lacunas [assim];
- **RAG híbrido (rubrica v1.3)**: a diretriz da categoria vem do campo `Categoria` do registro — garantia, não sorteio — e só o glossário é recuperado por similaridade;
- **Garantias em código**: enquadramento parseado da observação e injetado; fundo e artefatos de estúdio **não viajam** para a redação; cada artefato observado vira flag; a escala (maior dimensão) e a plausibilidade da medida saem do registro por aritmética; contradição entre campos é uma pergunta isolada, fora da tarefa de escrita.

O resultado se compara com o da v5 (mesmo modelo, sistema antigo), medido pela mesma régua: `avaliacao/checar_lote.py` roda as checagens de hoje sobre os lotes antigos.

*Metodologia: projeto construído com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~40–50 min**. Deixe a aba aberta durante a execução.

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 20 objetos, com salvaguardas de imagem (~3 min)

O lote: os **5 objetos do smoke test** (para comparar com os notebooks anteriores) + **15 novos**, sorteados com seed fixa (reproduzível) entre os itens que **não** estão no conjunto de avaliação — o lote serve para testar o pipeline em escala, sem "viciar" nos itens que depois vão dar a nota.

Duas salvaguardas novas ao baixar cada foto:

- **Orientação EXIF**: fotos de câmera guardam a rotação numa etiqueta interna que os navegadores aplicam, mas o Python não — sem esta linha, o modelo poderia receber uma foto deitada sem ninguém saber. `ImageOps.exif_transpose` aplica a rotação correta.
- **Conversão para RGB**: garante que qualquer foto (escala de cinza, outros formatos de cor) chegue ao modelo no formato esperado.

Desta vez o registro completo de cada objeto (povo, materiais, dimensões, descrição curatorial...) **viaja junto** — a correção do bug do Notebook 03.

In [ ]:
import io, re, requests
from PIL import Image, ImageOps

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
IDS_SMOKE = [9196, 665, 51023, 63283, 78838]
# 15 novos: sorteio seed 42, estratificado por categoria, excluindo os 50 casos
# de avaliação (seleção documentada no repositório, commit da E7)
IDS_LOTE = [1376, 84811, 883523, 2081, 5011, 200648, 210680, 5146, 500179, 3411, 1366, 4156, 205095, 905, 500322]

CAMPOS_REGISTRO = ["Nome do item", "Povo", "Categoria", "Matéria-prima",
                   "Técnica de confecção", "Dimensões", "Função",
                   "Estado de origem", "Ano de aquisição do objeto", "Descrição"]

objetos = []
for item_id in IDS_SMOKE + IDS_LOTE:
    item = requests.get(f"{BASE}/items/{item_id}", timeout=60).json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))
    foto = ImageOps.exif_transpose(foto).convert("RGB")  # salvaguardas
    meta_bruto = requests.get(f"{BASE}/item/{item_id}/metadata", timeout=60).json()
    todos = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    registro = {c: todos.get(c, "") for c in CAMPOS_REGISTRO}
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto, "registro": registro})
    print(f"✓ {item_id} — {item['title']} ({registro['Povo']})")
print(f"{len(objetos)} objetos carregados")

In [ ]:
# Etapa 3 — Rubrica v1.3 + RAG híbrido: categoria garantida + glossário recuperado
import json, os, requests
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
RUBRICA_DRIVE = f"{PROJETO}/dados/rubrica_v1_3.json"
RUBRICA_REPO = ("https://raw.githubusercontent.com/eduardotosto/acervo-que-fala/"
                "main/dados/rubrica/rubrica.json")
if os.path.exists(RUBRICA_DRIVE):
    with open(RUBRICA_DRIVE, encoding="utf-8") as f:
        rubrica = json.load(f)
    print(f"rubrica lida do Drive ✓")
else:
    rubrica = requests.get(RUBRICA_REPO, timeout=60).json()
    print("rubrica lida do repositório público (ainda não está no Drive) ✓")

trechos = rubrica["trechos"]
glossario = [t for t in trechos if t["categoria"] == "glossario"]
trechos_categoria = [t for t in trechos if t["categoria"] != "glossario"]
por_categoria = {}
for t in trechos_categoria:
    por_categoria.setdefault(t["categoria"], []).append(t)

# Embedder na CPU: 23 trechos e 20 consultas são trabalho trivial, e a T4 fica inteira
# para o modelo 4-bit (armadilha conhecida: device_map="auto" com a GPU já ocupada
# manda camadas para a CPU e quebra a geração).
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cpu")
vet_glossario = embedder.encode([t["texto"] for t in glossario], convert_to_tensor=True)
vet_categoria = embedder.encode([t["texto"] for t in trechos_categoria], convert_to_tensor=True)
# A família Qwen3-Embedding é instruída: a CONSULTA vai com o prompt de query, os
# documentos vão sem. Se a versão instalada não trouxer o prompt, seguimos sem ele.
PROMPT_QUERY = "query" if "query" in (getattr(embedder, "prompts", None) or {}) else None

def recuperar(categoria, consulta, k_glossario=2, k_fallback=2):
    """RAG híbrido. A diretriz da categoria vem do REGISTRO (o campo Categoria existe em
    100% dos itens) — é garantia, não sorteio; o glossário vem da busca semântica, que é
    onde a recuperação realmente agrega. Categoria fora da rubrica cai no modo semântico."""
    v = embedder.encode(consulta, prompt_name=PROMPT_QUERY, convert_to_tensor=True)
    scores = util.cos_sim(v, vet_glossario)[0]
    achados = [glossario[i] for i in scores.argsort(descending=True)[:k_glossario].tolist()]
    fixos = por_categoria.get(categoria)
    if fixos is None:
        s2 = util.cos_sim(v, vet_categoria)[0]
        fixos = [trechos_categoria[i] for i in s2.argsort(descending=True)[:k_fallback].tolist()]
    return fixos + achados

print(f"rubrica {rubrica['versao']}: {len(por_categoria)} categorias + {len(glossario)} "
      f"trechos de glossário | prompt de query: {PROMPT_QUERY or 'indisponível'} ✓")

In [ ]:
# Etapa 4 — Modelo (Qwen3-VL-8B em 4-bit, como nos notebooks anteriores)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("modelo carregado ✓")

## Etapa 5 — Observação v3: seções nomeadas (~15 min)

A observação deixou de ser texto corrido e virou um **formulário de seções** — cada uma responde uma pergunta neutra ("somente se existirem"), sem induzir resposta.

Três seções são **consumidas pelo código** e não chegam à redação:

- `ENQUADRAMENTO:` — vira uma decisão pronta, injetada na redação como variável (fim do "Detalhe de..." em objeto inteiro);
- `ARTEFATOS:` — cada item vira uma flag automaticamente (recall garantido, sem depender de o modelo lembrar);
- `FUNDO E ESTÚDIO:` — sai inteira. Mandar "há um fundo branco e uma etiqueta" junto com a ordem de ignorá-los é plantar a palavra proibida no contexto: foi assim que "sobre fundo X" sobreviveu em 15 dos 20 alts da v5, apesar da regra. O que o texto não pode usar, não viaja.

O parse aceita o cabeçalho em negrito (`**ENQUADRAMENTO:**`), formato que o modelo usa com frequência — sem isso, a seção era removida do texto mas não era lida, e o enquadramento caía no valor padrão sem ninguém perceber.

In [ ]:
PROMPT_OBSERVACAO_V3 = """Você está diante da fotografia de um objeto do acervo de um museu — objetos etnográficos de povos indígenas do Brasil, fotografados em estúdio. Este contexto serve para você reconhecer materiais e situações de estúdio; NÃO use para adivinhar o que o objeto é ou significa. Descreva somente o que está visível NESTA fotografia.

Preencha as seções abaixo, nesta ordem:

OBJETO: o que se vê, em uma frase — forma geral, sem nomear função nem significado.
MATERIAIS E CORES: os materiais aparentes e suas cores, do maior para o menor. Material que não dá para identificar recebe o termo genérico ("fibra", "madeira clara") — nunca chute espécie ou origem.
PADRÕES E TEXTURAS: desenhos, tramas e acabamentos visíveis, descritos pela forma (linhas, xadrez, diagonais) — somente se existirem.
PARTES E QUANTIDADES: partes distinguíveis e contáveis (tubos, furos, alças, penas destacadas).
POSIÇÃO: como o objeto está na foto (de pé, deitado, inclinado) e partes internas visíveis (boca, interior, verso).
LEGIBILIDADE: o que estiver ilegível ou incerto — declare a incerteza em vez de estimar.
FUNDO E ESTÚDIO: o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).
ENQUADRAMENTO: inteiro OU detalhe — "detalhe" SÓ se a foto mostra claramente apenas parte do objeto; objeto que encosta ou sangra nas margens conta como inteiro.
ARTEFATOS: os artefatos de estúdio vistos, separados por vírgula, ou "nenhum".

Regra geral: o que não está visível não existe para esta descrição. Responda em português."""

for n, obj in enumerate(objetos, 1):
    obj["observacao"] = gerar(
        [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V3}],
        max_tokens=500,
    )
    print(f"[{n}/{len(objetos)}] {obj['titulo']} observado ✓")

# --- Parse das seções: o código consome ENQUADRAMENTO, ARTEFATOS e FUNDO E ESTÚDIO ---
CABECALHO = r"(?:OBJETO|MATERIAIS E CORES|PADRÕES E TEXTURAS|PARTES E QUANTIDADES|POSIÇÃO|LEGIBILIDADE|FUNDO E ESTÚDIO|ENQUADRAMENTO|ARTEFATOS)"
CONSUMIDAS = r"(?:ENQUADRAMENTO|ARTEFATOS|FUNDO E ESTÚDIO)"
# O cabeçalho volta do modelo em quatro formatos ("ARTEFATOS:", "**ARTEFATOS:**",
# "**ARTEFATOS**:", "### ARTEFATOS:"). P cobre os quatro: sem ele a seção era removida
# do texto mas não era lida, e o enquadramento caía no valor padrão silenciosamente.
P = r"[#*]*\s*"

def secao(texto, nome):
    """Devolve o conteúdo de uma seção nomeada da observação, ou string vazia."""
    m = re.search(rf"{nome}{P}:{P}(.+?)(?=\n{P}{CABECALHO}{P}:|\Z)", texto, re.S | re.I)
    return m.group(1).strip().strip("*#").strip() if m else ""

for obj in objetos:
    obs = obj["observacao"]
    enq = secao(obs, "ENQUADRAMENTO").lower()
    obj["enquadramento"] = "detalhe" if enq.startswith("detalhe") else "inteiro"
    obj["enquadramento_ok"] = enq.startswith(("inteiro", "detalhe"))
    art = secao(obs, "ARTEFATOS")
    obj["artefatos_obs"] = [] if (not art or art.lower().startswith("nenhum")) else [
        a.strip(" .") for a in art.split(",") if a.strip(" .")
    ]
    # A redação recebe a observação SEM as três seções que o código já consumiu. FUNDO E
    # ESTÚDIO sai junto: mandar "existe um fundo branco e uma etiqueta" e pedir que o
    # modelo ignore é plantar a palavra proibida no contexto — o mecanismo que deixou o
    # fundo do estúdio em 15 dos 20 alts da v5, apesar da regra que o proibia. O que o
    # texto não pode usar, não viaja.
    obj["observacao_para_redacao"] = re.sub(
        rf"\n?{P}{CONSUMIDAS}{P}:.*?(?=\n{P}{CABECALHO}{P}:|\Z)", "", obs, flags=re.S | re.I).strip()
    # Consulta do RAG (só o glossário é recuperado): identidade + o que a foto mostra
    obj["consulta_rag"] = (f"{obj['titulo']}. {secao(obs, 'OBJETO')} "
                           f"{secao(obs, 'PADRÕES E TEXTURAS')}")[:400]

ok = sum(1 for o in objetos if o["enquadramento_ok"])
det = sum(1 for o in objetos if o["enquadramento"] == "detalhe")
vazou = sum(1 for o in objetos if re.search(r"(?<![a-zà-ú])(fundo|etiqueta|cartela)",
                                            o["observacao_para_redacao"], re.I))
print(f"\nparse: {ok}/{len(objetos)} com ENQUADRAMENTO válido ({det} 'detalhe') | "
      f"{sum(len(o['artefatos_obs']) for o in objetos)} artefatos observados | "
      f"{vazou} observações ainda citam fundo/etiqueta fora das seções consumidas")

## Etapa 5b — o que o código extrai do registro (~4 min)

Três defeitos resistiram a todas as versões do prompt: a **escala** saiu pela medida de uma parte (o bocal da zarabatana, não o comprimento), a **miniatura** nunca foi declarada, e `metadado_suspeito` ficou em **zero** por três lotes seguidos — inclusive no abano de 290 cm, que o modelo reproduziu como "cerca de 3 metros" sem estranhar.

Nenhum dos três é tarefa de escrita: são aritmética e comparação. Esta célula resolve os três em código, seguindo a regra do projeto — *o que precisa de 100% de garantia não se pede a um modelo*.

- **Escala**: a maior dimensão do registro, com o rótulo, ignorando medidas "com cordel/alça esticada" (que medem o objeto pendurado). A redação recebe a frase pronta e é proibida de recalcular.
- **Plausibilidade**: o teto de cada categoria é **Q3 + 3×IQR das dimensões do próprio acervo** (547 itens com medida parseável), com piso de 150 cm. É detecção de outlier sobre a distribuição real, não um número escolhido a dedo: dispara em 2 dos 547 itens — o abano de 290 cm e uma capa de pele de onça de 223 cm, que é grande de verdade. Flag é pedido de conferência, não veredito.
- **Contradição entre campos**: continua sendo trabalho de modelo, mas como **pergunta isolada** — só o registro, uma pergunta, resposta sim/não. Pedida no meio da redação, ela deu recall zero; sozinha, é uma tarefa que o modelo sabe fazer.

In [ ]:
# Etapa 5b — o que o CÓDIGO extrai do registro (aritmética não se pede a modelo).
# Este bloco é o mesmo texto de avaliacao/checar_lote.py, para que o notebook e a
# ferramenta que mede os lotes antigos nunca divirjam.
import collections

ROTULOS = r"(comprimento|altura|largura|di[âa]metro|espessura|profundidade)"
RE_ROTULO, RE_NUM = re.compile(ROTULOS, re.I), re.compile(r"\d+(?:[.,]\d+)?")
# medida "com cordel / com a alça esticada" mede o objeto pendurado, não a peça
RE_COM_CORDA = re.compile(r"com\s+(a\s+|o\s+)?(cordel|cord[aã]o|al[çc]a|amarra)|esticad", re.I)

# Teto de plausibilidade por categoria = Q3 + 3xIQR das dimensões do PRÓPRIO acervo
# (547 dos 555 itens de dados/itens.json têm medida parseável), com piso de 150 cm —
# abaixo disso, peça grande é plausível em qualquer categoria. No acervo inteiro o teto
# dispara 2 vezes: o abano de 290 cm (dimensão improvável, caso conhecido do projeto) e
# a capa de pele de onça de 223 cm (peça genuinamente grande). Flag é pedido de
# conferência humana, não veredito — a taxa de alarme é o que importa, e é de 0,4%.
TETOS_CATEGORIA = {
    "Adornos Plumários": 156,
    "Adornos de Materiais Ecléticos, Indumentária e Toucador": 106,
    "Armas": 491,
    "Cerâmica": 56,
    "Cordões e Tecidos": 103,
    "Etnobotânica": 33,
    "Instrumentos musicais e de sinalização": 76,
    "Objetos rituais, mágicos e lúdicos": 144,
    "Trançados": 181,
    "Utensílios e implementos de materiais ecléticos": 75,
}
PISO_SUSPEITA = 150
# miniatura: peça bem menor que o comum da categoria, em categorias de objeto grande
MEDIANAS_CATEGORIA = {"Cerâmica": 14.0, "Trançados": 41.5, "Armas": 212.5,
                      "Instrumentos musicais e de sinalização": 41.0,
                      "Utensílios e implementos de materiais ecléticos": 19.5}

def escala_do_registro(dimensoes):
    """Maior dimensão do objeto, em cm, com o rótulo — a regra editorial 25 ("escala é a
    maior dimensão, nunca a medida de uma parte"), resolvida em código. Trata a lista
    enumerada ("29,5; 21,5; ... e 9,5 cm de comprimento") herdando o rótulo do segmento
    seguinte. Devolve (valor, rótulo) ou None."""
    if not dimensoes or "cm" not in dimensoes.lower():
        return None
    segmentos = [s for s in re.split(r";|\s-\s", dimensoes) if s.strip()]
    candidatos = []
    for i, seg in enumerate(segmentos):
        if RE_COM_CORDA.search(seg):
            continue
        rot = RE_ROTULO.search(seg) or next(
            (RE_ROTULO.search(s) for s in segmentos[i + 1:] if RE_ROTULO.search(s)), None)
        rotulo = rot.group(1).lower() if rot else ""
        candidatos += [(float(m.group(0).replace(",", ".")), rotulo) for m in RE_NUM.finditer(seg)]
    return max(candidatos) if candidatos else None


def numero_pt(v):
    return f"{v:.0f}" if v >= 20 else f"{v:.1f}".replace(".", ",").replace(",0", "")


def analisar_registro(registro):
    """Devolve a linha ESCALA pronta para o prompt + as flags de metadado que a
    aritmética já resolve (dimensão fora do teto da categoria, ano impossível)."""
    cat, flags = registro.get("Categoria", ""), []
    e = escala_do_registro(registro.get("Dimensões", ""))
    if not e:
        escala = "não informada no registro — não escreva medida nenhuma"
    else:
        valor, rotulo = e
        escala = f"cerca de {numero_pt(valor)} cm" + (f" de {rotulo}" if rotulo else "")
        mediana = MEDIANAS_CATEGORIA.get(cat)
        if mediana and valor < 10 and valor < mediana / 2:
            escala += " — miniatura (bem menor que o comum na categoria): diga que é miniatura"
        teto = TETOS_CATEGORIA.get(cat)
        if teto and valor > max(teto, PISO_SUSPEITA):
            flags.append({"tipo": "metadado_suspeito", "detalhe":
                          f"{numero_pt(valor)} cm de {rotulo or 'dimensão'} está acima do teto de "
                          f"plausibilidade da categoria {cat} ({teto} cm, calculado do acervo)"})
    ano = (registro.get("Ano de aquisição do objeto") or "").strip()
    if ano.isdigit() and not (1850 <= int(ano) <= 2026):
        flags.append({"tipo": "metadado_suspeito", "detalhe": f"ano de aquisição improvável: {ano}"})
    return escala, flags

# Contradição entre campos do registro: pergunta ISOLADA, fora da tarefa de escrita.
# Pedida no meio da redação, ela deu recall zero em três lotes seguidos (v2, v4, v5).
PROMPT_CONTRADICAO = """Leia os campos do registro de catálogo de um museu abaixo e responda se existe CONTRADIÇÃO INTERNA: dois campos que afirmam coisas incompatíveis sobre o MESMO objeto.

Exemplo de contradição: o campo Descrição diz "brinquedo em miniatura" e o campo Função diz "utilizado para caça" — o mesmo objeto não é as duas coisas.
Não é contradição: campo vazio, informação que falta, detalhe que só um campo traz, ou estilo de escrita.

REGISTRO:
{registro}

Responda APENAS com JSON: {{"contradicao": true ou false, "campos": ["campo A", "campo B"], "detalhe": "uma frase explicando"}}"""

for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    obj["escala"], obj["flags_registro"] = analisar_registro(obj["registro"])
    try:
        r = extrair_json(gerar([{"type": "text", "text":
                                 PROMPT_CONTRADICAO.format(registro=registro_txt)}], max_tokens=250))
        obj["contradicao"] = r
        if r.get("contradicao") and r.get("detalhe"):
            obj["flags_registro"].append({"tipo": "metadado_suspeito", "detalhe":
                                          f"contradição no registro: {r['detalhe']}"})
    except Exception:
        obj["contradicao"] = None
    marca = f" | {len(obj['flags_registro'])} flag(s)" if obj["flags_registro"] else ""
    print(f"[{n}/{len(objetos)}] {obj['id']} escala: {obj['escala'][:52]}{marca}")

print(f"\n{sum(len(o['flags_registro']) for o in objetos)} flags de metadado geradas pelo código "
      f"(dimensão fora do teto, ano impossível, contradição entre campos)")

## Etapa 6 — Redação v9: Contrato de Fontes (~20 min)

O prompt foi redesenhado da estrutura, não por acréscimo (análise em `avaliacao/analise_prompt_rubrica.md`):

- cada insumo declara **o que autoriza** (observação → aparência; registro → fatos com atribuição; diretrizes → vocabulário);
- o **Contrato de Fontes** fica na posição de maior prioridade (depois dos dados, antes das saídas): sem fonte → não escreva; incerteza herda-se da seção LEGIBILIDADE; divergência vira flag, nunca harmonização; exemplos com lacunas [assim], nunca copiáveis;
- as saídas viraram **checklist positivo** (o que um bom texto contém), com só as proibições irredutíveis;
- o modelo recebe **decisões prontas** (enquadramento, escala) e só produz a flag que exige ler os dois lados: a divergência entre foto e catálogo;
- o nível 2 ganhou **teto de escuta** (180 palavras): texto ouvido em voz alta cansa antes de texto lido.

Depois da geração, o código guarda o **alt bruto** ao lado do alt final. Sem isso, um "sobre fundo X" zerado não distingue o que o prompt resolveu do que o pós-processamento escondeu — e é exatamente essa diferença que o lote está medindo. O pós-processamento é conservador de propósito: em caso de dúvida ele não mexe, e a verificação acusa.

In [ ]:
PROMPT_REDACAO_V9 = """Você escreve descrições de acessibilidade para o acervo digital de um museu. Elas serão OUVIDAS por pessoas cegas, através de leitores de tela — escreva em linguagem cotidiana, com frases que funcionam no ouvido (ordem direta, sem parênteses longos), sem jargão de catálogo.

INSUMOS — cada um autoriza um tipo de informação:

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (autoriza: aparência — o que é visível):
{observacao}

ENQUADRAMENTO DECIDIDO NA OBSERVAÇÃO: {enquadramento}

ESCALA CALCULADA DO REGISTRO: {escala}

REGISTRO DO MUSEU (autoriza: fatos — sempre com atribuição; o título nomeia o objeto):
{registro}

DIRETRIZES PARA ESTE TIPO DE OBJETO (autorizam: vocabulário e o que observar nesta categoria):
{diretrizes}

CONTRATO DE FONTES — prevalece sobre qualquer outra regra:
1. Cada informação escrita precisa de fonte: visível na observação, escrita no registro (e então leva atribuição) ou vocabulário das diretrizes. Sem fonte, não escreva — omitir é sempre permitido; um texto curto e todo verificável vale mais que um completo com um palpite.
2. Incerteza se herda: o que estiver na seção LEGIBILIDADE da observação, ou vier com hesitação ("parece", "talvez", "possivelmente"), sai do texto ou vira o termo genérico — nunca vira afirmação.
3. Divergência não se resolve no texto: quando observação e registro conflitam (nome, cor, quantidade, material, dimensão), não escolha um lado nem harmonize — registre em flags como divergencia_imagem_catalogo; no texto, o fato não visível fica com o registro e a aparência fica com a observação.
4. Os exemplos deste prompt mostram a FORMA das frases, com lacunas [assim]; preencha sempre com o conteúdo deste objeto, nunca com as palavras do exemplo.

PRODUZA TRÊS SAÍDAS:

A) alt_text — o que a fotografia mostra, para quem não a vê.
   Uma frase, no máximo 30 palavras, começando pelo objeto (nomeado pelo TÍTULO do registro) e pelo povo.
   Contém: o material, quando natural e sem tingimento; as cores, onde a cor informa (penas, miçangas, pinturas, tingimentos); a forma dos padrões (faixas, xadrez, losangos, geométrico); as quantidades da seção PARTES E QUANTIDADES da observação.
   Quando a peça tem pintura ou decoração aplicada sobre a base, descreva nesta ordem: primeiro a decoração e suas cores, depois a base — "pintura [tipo] em [cores] sobre [material da base]". A base nunca aparece sozinha ("sobre a [material]") sem dizer o que está sobre ela.
   Se o ENQUADRAMENTO diz "detalhe", comece com "Detalhe de [objeto]"; se diz "inteiro", não mencione enquadramento nem orientação que não informa.
   Aves de penas: só as que o registro nomear, com a cor primeiro: "penas [cores] de [ave]".
   Não aparecem aqui: o fundo ou o estúdio da fotografia, artefato de inventário, palavra de catálogo, medida.

B) descricao_objeto — o objeto em si, para quem quer conhecê-lo além da foto. Dois parágrafos, no máximo 180 palavras somadas — texto ouvido em voz alta cansa antes de texto lido.
   1º: abre direto com o objeto e sua função quando ela acrescenta algo (caça, ritual, preparo) — nunca o óbvio. A primeira informação vinda do registro leva a marca de atribuição: "segundo o registro do museu", "o registro informa que" ou equivalente — e TODO fato do catálogo que não é visível na foto (função, técnica, origem, ano, medidas, decorações que o registro descreve) carrega marca de atribuição na própria frase, com formulação variada. Depois, a aparência: formas, materiais e padrões em palavras comuns, cada informação dita uma vez.
   2º: os demais fatos do catálogo em frases naturais ("adquirido em [ano]"). A escala é EXATAMENTE a que está em ESCALA CALCULADA DO REGISTRO, escrita em frase natural — não recalcule, não escolha outra medida do registro, não cite medida de parte; se lá estiver escrito que é miniatura, diga que é miniatura; se lá estiver escrito que não há medida, não escreva medida nenhuma. Aves das penas detalhadas conforme o registro; significado cultural só se estiver no registro.
   Este texto descreve o objeto, não a fotografia: posição, fundo, enquadramento e a própria foto não existem aqui; relações que dependem do ponto de vista viram relações da peça ("decrescentes", "em degraus").
   O que não existe no objeto simplesmente não é mencionado — nada de "sem [coisa]" ou "não há [coisa]".

C) flags — o que precisa de revisão humana. Os artefatos de estúdio e os metadados improváveis já foram registrados automaticamente pelo código; concentre-se no que só quem leu os dois lados percebe:
   - divergencia_imagem_catalogo: conflito entre o que a observação vê e o que o registro afirma — cor, quantidade, material, formato — ou objeto visto diferente do que o título nomeia (use o título no texto e registre a diferença aqui).
   Lista vazia [] se não houver nada.

Responda APENAS com JSON: {{"alt_text": "...", "descricao_objeto": "...", "flags": [{{"tipo": "...", "detalhe": "..."}}]}}"""

# Pós-processamento conservador: remove o fundo de estúdio residual SÓ quando o trecho
# termina em pontuação dentro de no máximo duas palavras ("..., sobre fundo bege." ✓).
# Em "sobre fundo bege e boca larga" ele não casa e não remove nada — amputar a frase
# seria pior que deixar passar, e o que sobra a verificação acusa.
RE_FUNDO = re.compile(
    r",?\s*\b(?:sobre|em|contra|com|num|no|sob)\s+(?:um |uma |o |a )?fundo\b"
    r"(?:\s+[a-zà-ú-]+){0,2}\s*(?=[,.;]|$)", re.I)

def pos_processar_alt(alt):
    alt = RE_FUNDO.sub("", alt)
    alt = re.sub(r",\s*,", ",", alt)
    alt = re.sub(r"\s{2,}", " ", alt).strip(" ,")
    if alt and not alt.endswith("."):
        alt += "."
    return alt

for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    achados = recuperar(obj["registro"]["Categoria"], obj["consulta_rag"])
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V9.format(
        observacao=obj["observacao_para_redacao"],
        enquadramento=obj["enquadramento"],
        escala=obj["escala"],
        registro=registro_txt,
        diretrizes="\n".join(f"- {t['texto']}" for t in achados),
    )
    resposta = gerar([{"type": "text", "text": prompt}], max_tokens=700)
    try:
        saida = extrair_json(resposta)
        # o alt bruto fica salvo: sem ele não dá para separar o que o prompt resolveu
        # do que o pós-processamento escondeu — e é justamente isso que este lote mede
        obj["alt_bruto"] = saida["alt_text"]
        obj["alt_text"] = pos_processar_alt(saida["alt_text"])
        obj["descricao_objeto"] = saida["descricao_objeto"]
        flags_auto = [{"tipo": "artefato_estudio", "detalhe": a} for a in obj["artefatos_obs"]]
        flags_modelo = [f for f in saida.get("flags", [])
                        if f.get("tipo") not in ("artefato_estudio", "metadado_suspeito")]
        obj["flags"] = flags_auto + obj["flags_registro"] + flags_modelo
        obj["json_valido"] = True
    except Exception:
        obj["alt_bruto"] = obj["alt_text"] = resposta
        obj["descricao_objeto"], obj["flags"] = "", list(obj["flags_registro"])
        obj["json_valido"] = False
    print(f"[{n}/{len(objetos)}] {obj['titulo']}: {obj['alt_text'][:80]}... | flags: {len(obj['flags'])}")

mexidos = sum(1 for o in objetos if o.get("alt_bruto") != o["alt_text"])
print(f"\n{mexidos}/{len(objetos)} alts precisaram do pós-processamento de fundo "
      f"(na v5, 15/20 traziam 'sobre fundo X' — aqui o número mede o que o prompt não resolveu)")

## Etapa 7 — Verificação automática

As checagens agora usam **fronteira de palavra**. A versão anterior procurava o termo como pedaço de texto: "parece" casava com "aparece", "fundo" com "profundo", "coração" com "decoração" — o mesmo casamento raso de texto que o projeto diagnosticou nos modelos estava na própria régua que os media.

Checagens herdadas: JSON válido; povo no alt; artefato nos textos; até 30 palavras; atribuição; frases-etiqueta; "foi aquisição em"; ausências; foto no nível 2; especulação; frases vazias; "fundo" no alt; qualidade das flags.

Novas na v6: **coerência de enquadramento** (o alt começa com "Detalhe" se e somente se a observação decidiu "detalhe"); **medidas** — toda medida escrita tem que existir no registro, e a escala tem que ser a maior, que é o teste do bocal da zarabatana; **miniatura declarada**; **teto de escuta** do nível 2.

A função de verificação pula sozinha as checagens cujo campo não existe no item. É o que permite medir os lotes antigos com o critério de hoje: `avaliacao/checar_lote.py`, no repositório, usa esta mesma função para comparar v5, Gemma e v6 na mesma régua.

In [ ]:
# Etapa 7 — Verificação automática. Este bloco é o mesmo texto de
# avaliacao/checar_lote.py, que aplica estas checagens aos lotes anteriores — é assim
# que a v6 e a v5 são comparadas pela mesma régua.
# Todos os termos são procurados com FRONTEIRA DE PALAVRA. A versão anterior usava
# "termo in texto": "parece" casava com "aparece", "fundo" com "profundo", "coração"
# com "decoração" — o mesmo casamento raso de texto que o projeto diagnosticou nos
# modelos aparecia na própria régua que os media.
B = lambda termos: re.compile(r"(?<![a-zà-úA-ZÀ-Ú])(?:" + "|".join(termos) + r")", re.I)
RE_ARTEFATO = B(["cartela", "paleta", "numeraç", "marcaç", "etiqueta", "régua", "suporte"])
RE_AUSENCIA = B(["não há", "sem etiqueta", "sem sinais", "sem evidência", "sem artefato",
                 "sem marca"])
RE_ESPECULACAO = B(["sugere", "sugerindo", "parece", "parecendo", "possivelmente", "talvez"])
RE_VAZIA = B(["porte médio", "uso prático", "uso frequente", "sinais de uso", "forma funcional",
              "forma é funcional"])
RE_FUNDO_TXT = B(["fundo"])
RE_FOTO = re.compile(r"(?<![a-zà-ú])(?:posicionad|enquadr|fotografia|[dn]a imagem|ao fundo|"
                     r"plano (?:médio|geral|fechado|aberto)|close|"
                     r"inclinad\w*\s+(?:levemente\s+)?(?:para\s+)?[aà]?\s*"
                     r"(?:direita|esquerda|frente|trás))", re.I)
# jargão de fotografia no alt: "close-up" e "plano médio" apareceram como
# vocabulário novo de enquadramento na v5, depois que a regra proibiu
# "inteiro/horizontal/vertical". "Detalhe de..." continua sendo a marca sancionada.
RE_JARGAO_FOTO = re.compile(r"(?<![a-zà-ú])(?:close|plano (?:médio|geral|fechado|aberto)|primeiro plano)", re.I)
ABERTURAS_ETIQUETA = ("o objeto é", "trata-se de")
RE_MEDIDA_TXT = re.compile(r"(\d+(?:[.,]\d+)?)\s*cm", re.I)


def tem_atribuicao(texto):
    return re.search(r"(?<![a-zà-ú])(registro|catálogo|catalogo)", texto or "", re.I) is not None


def verificar(item):
    """Devolve os problemas como pares (chave, detalhe). A chave é estável, para somar o
    mesmo problema entre lotes; o detalhe é o que muda de item para item.

    As checagens que dependem de campos que só a v6 produz (o enquadramento decidido na
    observação) são puladas quando o campo não existe — é o que permite medir os lotes
    antigos sem inventar dado que eles não têm. A escala, por ser função só do registro,
    vale para todos."""
    p = []
    alt = item.get("alt_text", "") or ""
    d = item.get("descricao_objeto", "") or ""
    a, dl = alt.lower(), d.lower().strip()
    registro = item.get("registro", {}) or {}

    if item.get("json_valido") is False:
        p.append(("json_invalido", ""))
    if "enquadramento_ok" in item and not item["enquadramento_ok"]:
        p.append(("obs_sem_enquadramento", ""))

    povo = (registro.get("Povo") or "").strip()
    if povo and povo.split()[0].lower() not in a:
        p.append(("povo_ausente_no_alt", povo))
    if RE_ARTEFATO.search(a):
        p.append(("artefato_no_alt", RE_ARTEFATO.search(a).group(0)))
    if RE_FUNDO_TXT.search(a):
        p.append(("fundo_no_alt", ""))
    if RE_JARGAO_FOTO.search(a):
        p.append(("jargao_de_foto_no_alt", RE_JARGAO_FOTO.search(a).group(0)))
    if len(alt.split()) > 30:
        p.append(("alt_longo", f"{len(alt.split())} palavras"))
    if item.get("enquadramento"):
        alt_detalhe = a.strip().startswith("detalhe")
        if alt_detalhe and item["enquadramento"] != "detalhe":
            p.append(("enquadramento_incoerente", "alt diz Detalhe, observação diz inteiro"))
        if not alt_detalhe and item["enquadramento"] == "detalhe":
            p.append(("enquadramento_incoerente", "observação diz detalhe, alt não marca"))
    return p + _verificar_nivel2(item, d, dl, a, registro)


def _verificar_nivel2(item, d, dl, a, registro):
    p = []
    if d:
        if not tem_atribuicao(d):
            p.append(("nivel2_sem_atribuicao", ""))
        if any(dl.startswith(ab) for ab in ABERTURAS_ETIQUETA):
            p.append(("frase_etiqueta", dl[:18]))
        if "a função é" in dl:
            p.append(("frase_etiqueta", "a função é"))
        if "aquisição em" in dl:
            p.append(("aquisicao_em", ""))
        if RE_ARTEFATO.search(dl):
            p.append(("artefato_no_nivel2", RE_ARTEFATO.search(dl).group(0)))
        if RE_AUSENCIA.search(dl):
            p.append(("afirmacao_de_ausencia", RE_AUSENCIA.search(dl).group(0)))
        if RE_FOTO.search(dl):
            p.append(("foto_no_nivel2", RE_FOTO.search(dl).group(0)))
        if len(d.split()) > 200:
            p.append(("nivel2_longo", f"{len(d.split())} palavras"))
        # medidas: toda medida escrita tem que estar no registro, e a escala é a maior
        nums_txt = [float(x.replace(",", ".")) for x in RE_MEDIDA_TXT.findall(d)]
        nums_reg = [float(x.replace(",", ".")) for x in
                    re.findall(r"\d+(?:[.,]\d+)?", registro.get("Dimensões", "") or "")]
        for t in nums_txt:
            if nums_reg and not any(abs(r - t) <= 1.0 for r in nums_reg):
                p.append(("medida_fora_do_registro", f"{t:g} cm"))
        escala = item.get("escala") or ""
        if escala and nums_txt:
            m = re.search(r"(\d+(?:[.,]\d+)?)", escala)
            if m:
                esc = float(m.group(1).replace(",", "."))
                if not any(abs(esc - t) <= 1.0 for t in nums_txt):
                    p.append(("escala_errada", f"a maior é {esc:g} cm"))
        if "miniatura" in escala and "miniatura" not in dl:
            p.append(("miniatura_nao_declarada", ""))

    for nome, texto in [("alt", a), ("nível 2", dl)]:
        if RE_ESPECULACAO.search(texto):
            p.append(("especulacao", f"{nome}: {RE_ESPECULACAO.search(texto).group(0)}"))
        if RE_VAZIA.search(texto):
            p.append(("frase_vazia", f"{nome}: {RE_VAZIA.search(texto).group(0)}"))

    for f in item.get("flags", []) or []:
        det = (f.get("detalhe") or "").lower()
        if (f.get("tipo") == "artefato_estudio" and RE_FUNDO_TXT.search(det)
                and not RE_ARTEFATO.search(det)):
            p.append(("flag_de_fundo", ""))
        if RE_AUSENCIA.search(det):
            p.append(("flag_de_ausencia", ""))
    return p

for obj in objetos:
    obj["problemas"] = verificar(obj)
    detalhe = "; ".join(f"{k}{' (' + v + ')' if v else ''}" for k, v in obj["problemas"])
    print(f"{obj['id']} {obj['titulo'][:28]:28} " + ("✓" if not obj["problemas"] else "⚠ " + detalhe))

abano = next(o for o in objetos if o["id"] == 63283)
checks_abano = {
    "enquadramento 'detalhe'": abano["enquadramento"] == "detalhe",
    "alt abre com 'Detalhe'": abano["alt_text"].strip().lower().startswith("detalhe"),
    "nível 2 com atribuição": tem_atribuicao(abano["descricao_objeto"]),
    "290 cm virou metadado_suspeito": any(f["tipo"] == "metadado_suspeito" for f in abano["flags"]),
}
print("\nCaso-referência Abano (63283): " +
      " | ".join(f"{k} {'✓' if v else '✗'}" for k, v in checks_abano.items()))
tipos = collections.Counter(f["tipo"] for o in objetos for f in o["flags"])
chaves = collections.Counter(k for o in objetos for k, _ in o["problemas"])
print(f"Total: {sum(1 for o in objetos if not o['problemas'])}/{len(objetos)} objetos sem problemas")
print(f"flags por tipo: {dict(tipos)}")
print(f"problemas por checagem: {dict(chaves)}")

In [ ]:
# Etapa 8 — Salvar no Drive (arquivo v6 — os resultados anteriores ficam preservados)
resultado = {
    "notebook": "04_pipeline_completo_v6",
    "modelo": MODELO,
    "embedding": "Qwen/Qwen3-Embedding-0.6B (CPU)",
    "rubrica_versao": rubrica["versao"],
    "rag": "híbrido: diretriz da categoria pelo registro + glossário por similaridade (k=2)",
    "prompt_observacao_v3": PROMPT_OBSERVACAO_V3,
    "prompt_redacao_v9": PROMPT_REDACAO_V9,
    "prompt_contradicao": PROMPT_CONTRADICAO,
    "tetos_categoria": TETOS_CATEGORIA,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "enquadramento": o["enquadramento"],
         "enquadramento_ok": o["enquadramento_ok"], "artefatos_obs": o["artefatos_obs"],
         "escala": o["escala"], "contradicao": o.get("contradicao"),
         "alt_bruto": o.get("alt_bruto", ""), "alt_text": o["alt_text"],
         "descricao_objeto": o["descricao_objeto"],
         "flags": o["flags"], "flags_registro": o["flags_registro"],
         "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/04_pipeline_completo_v6.json"
os.makedirs(os.path.dirname(destino), exist_ok=True)
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude que o Notebook 04 **v6** terminou — ele busca o resultado no Drive e roda a comparação decisiva: **v5 (sistema antigo) contra v6 (sistema redesenhado)**, mesmo modelo, mesma régua. Se os defeitos induzidos sumirem (o exemplo do prompt copiado para fora de contexto, "Detalhe" em objeto inteiro, papagaio de frases), a hipótese do autor estava certa: o gargalo era o sistema de instruções, não o modelo.

Três números a olhar primeiro, porque separam o que o prompt resolveu do que o código garantiu:

1. **alts que precisaram do pós-processamento** — mede o que o prompt v9 não resolveu sozinho; na v5, 15 dos 20 traziam o fundo do estúdio no alt;
2. **flags por tipo** — artefato e metadado suspeito agora vêm do código; a divergência entre foto e catálogo continua sendo trabalho do modelo, e é onde o Gemma tinha vantagem no bake-off;
3. **coerência de enquadramento e medidas** — as duas checagens novas, que só podiam existir depois de o código decidir enquadramento e escala.

Só depois dessa resposta o bake-off de redator volta à mesa.

**O que este notebook prova:** o efeito isolado do redesenho do sistema — observação estruturada, contrato de fontes e garantias em código — medido com o mesmo modelo e os mesmos 20 objetos. **O que ainda não prova:** as métricas nos 40 casos (E8) e a avaliação cega (E10).